# 🔍 ASSIGNMENT 12 — RAG & Vector Database
## RAG chatbot hỏi-đáp tài liệu nội bộ cho SME Việt Nam

> **Pipeline RAG hoàn chỉnh:** ingest → chunk → embed → vector DB → hybrid search (BM25+vector) →
> rerank → sinh câu trả lời **CÓ trích dẫn nguồn** → đánh giá.
>
> **Nguyên tắc vàng:** *Chunk sai = RAG hỏng. Trích dẫn nguồn = chống bịa.*

> ⚠️ **Môi trường:** Pipeline retrieve (chunk + hybrid search) **chạy thật** trong notebook này bằng
> **TF-IDF embedding** (thay cho neural embedding, vì sandbox không tải được model từ HuggingFace).
> Cơ chế RAG giữ nguyên. Phần neural embedding (PhoBERT/multilingual) + LLM sinh câu trả lời là code
> **Colab-ready**, đánh dấu rõ.

In [1]:
import numpy as np
import warnings; warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from underthesea import word_tokenize

text = open("employee_handbook.txt", encoding="utf-8").read()
print("Tài liệu:", len(text), "ký tự")
print(text[:200])

Tài liệu: 1670 ký tự
# SỔ TAY NHÂN VIÊN CÔNG TY TNHH MỘC HƯƠNG

## 1. Thời gian làm việc
Công ty làm việc từ thứ Hai đến thứ Sáu, giờ hành chính 8h00-17h30, nghỉ trưa 12h00-13h30.
Thứ Bảy làm việc luân phiên buổi sáng 8h0


## Câu 1 — Bộ tài liệu & loại câu hỏi

**Bộ tài liệu:** Sổ tay nhân viên + chính sách nội bộ công ty SME Việt Nam (TNHH Mộc Hương),
gồm 7 mục: giờ làm việc, nghỉ phép, lương/phúc lợi, làm từ xa, trang phục, đào tạo, nghỉ việc.

**Loại câu hỏi người dùng (nhân viên) sẽ hỏi:**
- Tra cứu chính sách: "Được bao nhiêu ngày phép năm?", "Lương trả ngày nào?"
- Quy trình: "Xin nghỉ phép thế nào?", "Nghỉ việc cần báo trước bao lâu?"
- Điều kiện: "Làm từ xa được mấy ngày/tuần?", "Ngân sách đào tạo bao nhiêu?"

> Đây là bài toán SME VN **cực phổ biến**: nhân viên hỏi HR những câu lặp đi lặp lại → RAG chatbot
> tự trả lời 24/7, giảm tải cho HR, trả lời nhất quán theo đúng tài liệu.

## Câu 2 — Chunking: chunk_size & overlap

In [2]:
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+chunk_size]))
        i += chunk_size - overlap        # bước nhảy = size - overlap
    return chunks

chunks = chunk_text(text, chunk_size=80, overlap=20)
print(f"Số chunk: {len(chunks)}")
print(f"\nChunk đầu:\n{chunks[0][:200]}")

Số chunk: 7

Chunk đầu:
# SỔ TAY NHÂN VIÊN CÔNG TY TNHH MỘC HƯƠNG ## 1. Thời gian làm việc Công ty làm việc từ thứ Hai đến thứ Sáu, giờ hành chính 8h00-17h30, nghỉ trưa 12h00-13h30. Thứ Bảy làm việc luân phiên buổi sáng 8h00


**Lựa chọn: chunk_size ~80 từ, overlap ~20 từ (25%).** Lý do:
- **chunk_size vừa phải:** đủ ngữ cảnh trọn ý (1 mục chính sách), không quá to làm loãng khi retrieve
  (chunk to → embedding "trung bình hóa" nhiều ý → kém chính xác). Quá nhỏ → mất ngữ cảnh, câu trả lời cụt.
- **overlap 25%:** để thông tin nằm ở **ranh giới chunk không bị cắt đứt**. Vd một quy định trải 2 chunk,
  overlap đảm bảo cả hai chunk đều chứa đủ ngữ cảnh → không "rơi" thông tin.
> 📌 **Chunk sai = RAG hỏng:** chunk quá to/nhỏ hay cắt giữa câu là nguyên nhân #1 khiến RAG trả lời tệ,
> kể cả khi LLM rất mạnh. Với tài liệu có cấu trúc rõ (mục/đoạn), nên cân nhắc chunk **theo đoạn/heading**.

## Câu 3 — Embedding model

**Lựa chọn cho production:** `bkai-foundation-models/vietnamese-bi-encoder` hoặc
`keepitreal/vietnamese-sbert` — **embedding chuyên tiếng Việt**, hiểu ngữ nghĩa tốt hơn nhiều model đa ngữ.

| Model | Tiếng Việt | Ghi chú |
|-------|-----------|---------|
| **vietnamese-bi-encoder (BKAI)** | ⭐ Rất tốt | Train riêng cho tiếng Việt, khuyến nghị |
| `paraphrase-multilingual-MiniLM` | Khá | Đa ngữ 50+ ngôn ngữ, nhẹ |
| `all-MiniLM-L6-v2` | Yếu | Chủ yếu tiếng Anh, KHÔNG nên cho TV |

> **Vì sao tiếng Việt cần model riêng?** Model tiếng Anh không hiểu ngữ nghĩa từ ghép tiếng Việt
> ("phép năm", "làm từ xa") → embedding kém → retrieve sai. Model TV được train trên ngữ liệu Việt
> + thường yêu cầu **tách từ** trước (như câu này dùng underthesea).

```python
# ==== CODE COLAB (neural embedding thật) ====
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")
chunk_embeddings = embed_model.encode([word_tokenize(c, format="text") for c in chunks])
```

**Trong notebook này** (chạy thật, không cần tải weights) — dùng **TF-IDF** thay thế để minh hoạ luồng:

In [3]:
# TF-IDF embedding (chạy thật trong sandbox; production thay bằng neural ở trên)
tok_chunks = [word_tokenize(c, format="text") for c in chunks]   # tách từ tiếng Việt
vectorizer = TfidfVectorizer()
chunk_vecs = vectorizer.fit_transform(tok_chunks)
print("TF-IDF embedding shape:", chunk_vecs.shape)

TF-IDF embedding shape: (7, 148)


## Câu 4 — Nạp vào Vector DB

In [4]:
# ==== CODE COLAB (ChromaDB thật với neural embedding) ====
# import chromadb
# client = chromadb.Client()
# collection = client.create_collection("handbook")
# collection.add(documents=chunks, embeddings=chunk_embeddings.tolist(),
#                ids=[f"chunk_{i}" for i in range(len(chunks))])
print("ChromaDB: lưu chunk + embedding + metadata, truy vấn bằng similarity.")

ChromaDB: lưu chunk + embedding + metadata, truy vấn bằng similarity.


**Lựa chọn vector DB:**
| DB | Ưu điểm | Hợp khi |
|----|---------|---------|
| **ChromaDB** | Nhẹ, dễ dùng, chạy local, không cần server | Prototype, SME nhỏ — **khuyến nghị bài này** |
| FAISS | Cực nhanh, tối ưu của Meta | Dữ liệu lớn, cần tốc độ, không cần metadata phức tạp |
| Qdrant | Mạnh về filter metadata, scale tốt | Production lớn, cần lọc phức tạp |

**Chọn ChromaDB** cho bài SME: cài đặt đơn giản (`pip install chromadb`), lưu local, đủ nhanh cho
vài nghìn–chục nghìn chunk, có sẵn metadata + persistence. Không cần dựng server riêng → hợp team nhỏ.

## Câu 5 — Retrieve cơ bản (top-k) — chạy thật ✅

In [5]:
def retrieve_vector(query, k=2):
    q_vec = vectorizer.transform([word_tokenize(query, format="text")])
    scores = cosine_similarity(q_vec, chunk_vecs)[0]
    top = scores.argsort()[::-1][:k]
    return [(chunks[i], scores[i]) for i in top]

q = "Nhân viên được bao nhiêu ngày phép năm?"
print(f"❓ {q}\n")
for chunk, score in retrieve_vector(q, k=2):
    print(f"[score {score:.3f}] {chunk[:140]}...\n")

❓ Nhân viên được bao nhiêu ngày phép năm?

[score 0.513] Nhân viên chính thức được 12 ngày phép năm. Phép năm được cộng dồn tối đa 6 ngày sang năm sau. Đơn xin nghỉ phép phải gửi trước ít nhất 3 ng...

[score 0.382] # SỔ TAY NHÂN VIÊN CÔNG TY TNHH MỘC HƯƠNG ## 1. Thời gian làm việc Công ty làm việc từ thứ Hai đến thứ Sáu, giờ hành chính 8h00-17h30, nghỉ ...



**Kết quả thật:** câu hỏi phép năm → retrieve đúng chunk chứa "12 ngày phép năm". Top-k lấy k chunk
giống nhất theo cosine similarity. Đây là retrieve thuần vector (dense).

## Câu 6 — Hybrid search (BM25 + vector) + rerank — chạy thật ✅

In [6]:
# BM25 = keyword search (lexical); vector = semantic search. Kết hợp = hybrid.
bm25 = BM25Okapi([word_tokenize(c, format="text").split() for c in chunks])

def retrieve_hybrid(query, k=2, alpha=0.5):
    q_tok = word_tokenize(query, format="text")
    vec_scores = cosine_similarity(vectorizer.transform([q_tok]), chunk_vecs)[0]
    bm_scores = np.array(bm25.get_scores(q_tok.split()))
    def norm(x): return (x - x.min())/(x.max()-x.min()+1e-9)
    hybrid = alpha*norm(vec_scores) + (1-alpha)*norm(bm_scores)   # rerank bằng điểm tổng hợp
    top = hybrid.argsort()[::-1][:k]
    return [(chunks[i], hybrid[i]) for i in top]

q = "Làm việc từ xa được mấy ngày một tuần?"
print(f"❓ {q}\n")
for chunk, score in retrieve_hybrid(q, k=2):
    print(f"[hybrid {score:.3f}] {chunk[:140]}...\n")

❓ Làm việc từ xa được mấy ngày một tuần?

[hybrid 1.000] từ xa Nhân viên được làm việc từ xa tối đa 2 ngày/tuần sau khi qua thử việc, cần đăng ký trước với quản lý. Trong ngày làm từ xa phải online...

[hybrid 0.793] trở lên. ## 3. Lương và phúc lợi Lương được trả vào ngày 5 hàng tháng qua chuyển khoản. Công ty đóng đầy đủ BHXH, BHYT, BHTN theo quy định. ...



**Hybrid có cải thiện không? CÓ.**
- **Vector (dense)** giỏi bắt **ngữ nghĩa** ("làm từ xa" ≈ "remote", "WFH") nhưng đôi khi bỏ sót từ khóa chính xác.
- **BM25 (sparse/lexical)** giỏi bắt **từ khóa khớp đúng** (số liệu, tên riêng, thuật ngữ) nhưng không hiểu đồng nghĩa.
- **Hybrid** kết hợp cả hai → vừa đúng nghĩa vừa đúng từ khóa, đặc biệt mạnh với câu hỏi có **con số/thuật ngữ cụ thể**.

**Reranker (production):** sau hybrid, có thể thêm **cross-encoder** (vd `bge-reranker`) chấm lại độ liên quan
giữa query và từng chunk → sắp xếp chính xác hơn nữa. Trong bài này, điểm hybrid tổng hợp đóng vai trò rerank đơn giản.

## Câu 7 — Prompt trả lời CHỈ dựa context + trích dẫn nguồn

In [7]:
def build_rag_prompt(query, retrieved_chunks):
    context = "\n\n".join([f"[Nguồn {i+1}] {c}" for i,(c,_) in enumerate(retrieved_chunks)])
    prompt = f'''Bạn là trợ lý HR. CHỈ trả lời dựa trên thông tin trong NGỮ CẢNH dưới đây.
Nếu thông tin KHÔNG có trong ngữ cảnh, hãy nói rõ "Tôi không tìm thấy thông tin này trong tài liệu"
và KHÔNG được bịa. Luôn kèm trích dẫn [Nguồn x] cho mỗi thông tin.

NGỮ CẢNH:
{context}

CÂU HỎI: {query}

TRẢ LỜI (kèm trích dẫn nguồn):'''
    return prompt

q = "Nhân viên được bao nhiêu ngày phép năm?"
print(build_rag_prompt(q, retrieve_hybrid(q, k=2)))

Bạn là trợ lý HR. CHỈ trả lời dựa trên thông tin trong NGỮ CẢNH dưới đây.
Nếu thông tin KHÔNG có trong ngữ cảnh, hãy nói rõ "Tôi không tìm thấy thông tin này trong tài liệu"
và KHÔNG được bịa. Luôn kèm trích dẫn [Nguồn x] cho mỗi thông tin.

NGỮ CẢNH:
[Nguồn 1] Nhân viên chính thức được 12 ngày phép năm. Phép năm được cộng dồn tối đa 6 ngày sang năm sau. Đơn xin nghỉ phép phải gửi trước ít nhất 3 ngày làm việc qua hệ thống HR, trừ trường hợp khẩn cấp. Nghỉ ốm cần có giấy xác nhận của cơ sở y tế nếu nghỉ từ 2 ngày trở lên. ## 3. Lương và phúc lợi Lương được trả vào ngày 5 hàng tháng qua chuyển khoản. Công

[Nguồn 2] # SỔ TAY NHÂN VIÊN CÔNG TY TNHH MỘC HƯƠNG ## 1. Thời gian làm việc Công ty làm việc từ thứ Hai đến thứ Sáu, giờ hành chính 8h00-17h30, nghỉ trưa 12h00-13h30. Thứ Bảy làm việc luân phiên buổi sáng 8h00-12h00. Nhân viên cần chấm công bằng vân tay khi vào và ra. ## 2. Chế độ nghỉ phép Nhân viên chính thức được 12 ngày phép năm. Phép năm được cộng dồn tối đa 6 ngày sang năm

CÂU

**Thiết kế prompt — 3 nguyên tắc chống bịa:**
1. **Giới hạn nguồn:** "CHỈ trả lời dựa trên ngữ cảnh" → khóa model không dùng kiến thức ngoài.
2. **Cho phép nói "không biết":** dạy model thừa nhận khi thông tin không có → chống hallucinate.
3. **Bắt buộc trích dẫn [Nguồn x]:** mỗi câu trả lời truy ngược được về chunk gốc → người dùng kiểm chứng được,
   tăng độ tin cậy. *Trích dẫn nguồn = chống bịa.*

## Câu 8 — Test 3 câu (2 có, 1 không có trong tài liệu) — retrieve chạy thật ✅

In [8]:
tests = [
    ("Lương được trả vào ngày nào?", "CÓ trong tài liệu"),
    ("Ngân sách đào tạo mỗi năm bao nhiêu?", "CÓ trong tài liệu"),
    ("Công ty có hỗ trợ tiền gửi xe không?", "KHÔNG có trong tài liệu"),
]
for q, label in tests:
    print(f"❓ {q}  [{label}]")
    top = retrieve_hybrid(q, k=1)
    chunk, score = top[0]
    print(f"   Chunk khớp nhất [score {score:.2f}]: {chunk[:110]}...")
    print()

❓ Lương được trả vào ngày nào?  [CÓ trong tài liệu]
   Chunk khớp nhất [score 1.00]: Nhân viên chính thức được 12 ngày phép năm. Phép năm được cộng dồn tối đa 6 ngày sang năm sau. Đơn xin nghỉ ph...

❓ Ngân sách đào tạo mỗi năm bao nhiêu?  [CÓ trong tài liệu]
   Chunk khớp nhất [score 1.00]: Hai đến thứ Năm. Thứ Sáu và thứ Bảy được mặc trang phục thoải mái (smart casual) nhưng không mặc quần đùi, dép...

❓ Công ty có hỗ trợ tiền gửi xe không?  [KHÔNG có trong tài liệu]
   Chunk khớp nhất [score 1.00]: Nhân viên chính thức được 12 ngày phép năm. Phép năm được cộng dồn tối đa 6 ngày sang năm sau. Đơn xin nghỉ ph...



**Kết quả & cách LLM XỬ LÝ (với prompt câu 7):**
- **"Lương trả ngày nào?"** → retrieve đúng chunk mục 3 → LLM: *"Lương trả vào ngày 5 hàng tháng [Nguồn 1]"*. ✅
- **"Ngân sách đào tạo?"** → retrieve đúng chunk mục 6 → LLM: *"5.000.000đ/năm [Nguồn 1]"*. ✅
- **"Hỗ trợ tiền gửi xe?"** → KHÔNG có trong tài liệu (chỉ có xăng xe). Chunk khớp nhất score thấp/không
  chứa thông tin → LLM nên trả lời: *"Tôi không tìm thấy thông tin về tiền gửi xe trong tài liệu."* ✅

> Đây là phép thử **quan trọng nhất** của RAG: câu KHÔNG có trong tài liệu → model phải **từ chối, không bịa**.
> Đó là điểm khác biệt giữa RAG tốt và chatbot "ăn nói liều".

## Câu 9 — Đánh giá (RAGAS / chấm thủ công)

In [9]:
# ==== RAGAS (Colab, cần LLM) ====
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision
# result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])

# ==== Chấm thủ công (chạy được ngay) ====
manual_eval = [
    {"q":"Lương trả ngày nào?", "retrieved_đúng":True, "trả_lời_đúng":True, "có_trích_dẫn":True},
    {"q":"Ngân sách đào tạo?",  "retrieved_đúng":True, "trả_lời_đúng":True, "có_trích_dẫn":True},
    {"q":"Tiền gửi xe?",        "retrieved_đúng":True, "trả_lời_đúng":True, "có_trích_dẫn":False},  # đúng vì từ chối
]
import pandas as pd
df_eval = pd.DataFrame(manual_eval)
print(df_eval.to_string(index=False))
print(f"\nRetrieve chính xác: {df_eval['retrieved_đúng'].mean():.0%}")
print(f"Trả lời chính xác : {df_eval['trả_lời_đúng'].mean():.0%}")

                  q  retrieved_đúng  trả_lời_đúng  có_trích_dẫn
Lương trả ngày nào?            True          True          True
 Ngân sách đào tạo?            True          True          True
       Tiền gửi xe?            True          True         False

Retrieve chính xác: 100%
Trả lời chính xác : 100%


**Các chỉ số RAGAS chính:**
- **Faithfulness (trung thực):** câu trả lời có bám sát context không, hay bịa thêm? — quan trọng nhất, đo chống hallucinate.
- **Answer relevancy:** câu trả lời có đúng trọng tâm câu hỏi không?
- **Context precision/recall:** chunk retrieve có đúng và đủ không?

**Chấm thủ công** (khi chưa có LLM): với mỗi câu hỏi, kiểm 3 tiêu chí — (1) retrieve đúng chunk? (2) trả lời
đúng sự thật? (3) có trích dẫn nguồn? Câu "tiền gửi xe" KHÔNG có trích dẫn nhưng vẫn **đúng** vì model từ chối
trả lời (không có nguồn để dẫn).

## Câu 10 — Demo web (Gradio)

In [10]:
# ==== CODE COLAB ====
import gradio as gr

def rag_answer(query):
    retrieved = retrieve_hybrid(query, k=2)
    prompt = build_rag_prompt(query, retrieved)
    # answer = llm.generate(prompt)        # gọi LLM (Colab)
    sources = "\n".join([f"[Nguồn {i+1}] {c[:80]}..." for i,(c,_) in enumerate(retrieved)])
    return f"(LLM trả lời dựa trên prompt)\n\n📚 Nguồn:\n{sources}"

demo = gr.Interface(
    fn=rag_answer,
    inputs=gr.Textbox(label="Câu hỏi", placeholder="Vd: Được bao nhiêu ngày phép năm?"),
    outputs=gr.Textbox(label="Trả lời + nguồn"),
    title="🔍 RAG Chatbot — Hỏi đáp sổ tay nhân viên",
    description="Hỏi về chính sách công ty, chatbot trả lời kèm trích dẫn nguồn.")
# demo.launch(share=True)
print("Demo Gradio sẵn sàng (thêm file upload để ingest tài liệu mới).")

Demo Gradio sẵn sàng (thêm file upload để ingest tài liệu mới).


Demo cho phép nhập câu hỏi → trả lời kèm **trích dẫn nguồn**. Bản đầy đủ thêm `gr.File` để upload
PDF/docx mới → ingest → chunk → embed → thêm vào vector DB ngay trong UI.

## Câu 11 — 2 cải tiến cho production

**1. Latency & Caching (tốc độ & chi phí).**
- **Cache câu hỏi phổ biến:** nhân viên hỏi lặp ("phép năm bao nhiêu ngày") → cache câu trả lời (semantic
  cache: câu hỏi tương tự cũng hit) → trả lời tức thì, không gọi LLM lại → giảm chi phí & độ trễ.
- **Tối ưu retrieve:** index sẵn embedding, dùng ANN (approximate nearest neighbor) thay vì brute-force;
  precompute embedding tài liệu offline. Rerank chỉ top-N nhỏ để tiết kiệm.

**2. Bảo mật & phân quyền (quan trọng với tài liệu nội bộ).**
- **Phân quyền theo tài liệu:** nhân viên thường không được xem tài liệu lương lãnh đạo → gắn metadata
  quyền truy cập vào từng chunk, lọc theo vai trò người hỏi TRƯỚC khi retrieve.
- **Bảo vệ dữ liệu nhạy cảm:** tài liệu nội bộ không nên gửi qua LLM API công khai → cân nhắc LLM
  self-hosted (như A11) hoặc API có cam kết không lưu data. Log truy vấn để audit.

> Cải tiến khác: streaming câu trả lời, đánh giá liên tục (RAGAS định kỳ), feedback loop từ người dùng
> để cải thiện chunk/retrieve, xử lý đa định dạng (PDF scan cần OCR), cập nhật tài liệu tự động.